# Phase 3, step 2 — categorical columns become usable

Step 1 ended with a working run, and a quiet failure baked into it: `city` was
**dropped**. `train_tabular_model` feeds a GBDT a numeric matrix, so every string
column silently left the building, and the model never saw it.

This notebook closes that hole with a fourth tool, `fit_encoding`. As before you
drive everything **by hand** first — library level, then tool level, then the agent.

Three things to take away:

1. **Which encoder for which column** — and why "just use integers" is a trap.
2. **Out-of-fold vs full-fit** — the one place leakage can enter through the front door.
3. **Lineage** — how a fitted encoder survives all the way into the deployable bundle.

## 1. The trap: label encoding

The obvious move with a string column is to replace each category with a number:

```python
df["size"] = pd.factorize(df["size"])[0]          # or
df["size"] = df["size"].astype("category").cat.codes
```

This is not a leakage bug — it never looks at the target. It has two *other* problems,
and the second one is the dangerous one.

**Problem 1: invented order.** `{"blue": 0, "green": 1, "red": 2}` tells the model green
sits between blue and red. A tree can undo that with extra splits, so it mostly costs
you sample-efficiency. A linear model just believes it.

**Problem 2: the mapping isn't state.** `factorize` orders by *first appearance*;
`.cat.codes` orders *lexicographically*. Either way the codes come from **whichever
DataFrame you happened to call it on**. Call it on train, call it again on test, and if
the rows arrive in a different order or a category is missing, the two frames disagree
about what `1` means. Nothing raises. The model just gets worse for no visible reason.

That second problem is why encoders in this library are **objects that are fitted and
serialized**, not one-line casts.

## 2. Setup — a dataset with both kinds of categorical

- `city` — **nominal**. berlin/hamburg/koeln have no order.
- `size` — **ordinal**. small < medium < large is real, and it drives the target.

In [ ]:
import os, json, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 300

size = rng.choice(["small", "medium", "large"], size=n)
size_rank = pd.Series(size).map({"small": 0, "medium": 1, "large": 2}).to_numpy()
city = rng.choice(["berlin", "hamburg", "koeln"], size=n)
city_effect = pd.Series(city).map({"berlin": 0.8, "hamburg": -0.4, "koeln": 0.0}).to_numpy()
x1 = rng.normal(size=n)

df = pd.DataFrame({
    "x1": x1,
    "city": city,
    "size": size,
    "y": (0.9 * size_rank + city_effect + 0.5 * x1 + rng.normal(scale=0.5, size=n) > 1.0).astype(int),
})

workdir = tempfile.mkdtemp(prefix="harness_enc_")
data_path = os.path.join(workdir, "data.csv")
df.to_csv(data_path, index=False)

df.head()

## 3. Watch the trap spring

Same column, same categories — two frames that disagree about what `1` means.

In [ ]:
train = df.head(50)[["size"]]
serve = df.tail(50)[["size"]].reset_index(drop=True)

train_codes = pd.factorize(train["size"])[0]
serve_codes = pd.factorize(serve["size"])[0]

train_map = dict(zip(train["size"], train_codes))
serve_map = dict(zip(serve["size"], serve_codes))

print("train mapping:", train_map)
print("serve mapping:", serve_map)
print("agree?       :", train_map == serve_map)

If those two dicts differ, a model trained on the first and served the second is being
fed a column whose meaning changed underneath it — and nothing in the stack complains.
Even when they happen to agree, they agree *by luck of row order*.

**Check yourself:** why can't you fix this by sorting the categories alphabetically
before assigning codes?

> Because alphabetical order is not the *semantic* order. `large, medium, small` sorts
> to `0, 1, 2` — exactly backwards. It makes the mapping stable, which fixes problem 2,
> while making problem 1 worse: now the invented order is confidently wrong.

## 4. The encoders, at library level

Three, and the choice is about the column, not about taste:

| Encoder | Uses `y`? | Use it when |
|---|---|---|
| `OrdinalEncoder` | no | the categories have a **real order** you can state |
| `FrequencyEncoder` | no | nominal; you want a cheap, safe signal (how common) |
| `TargetEncoder` | **yes** | nominal, enough rows per category to estimate a mean |

Only the third can leak. Start with the first.

In [ ]:
from tabular_agent.encoders import OrdinalEncoder

enc = OrdinalEncoder({"size": ["small", "medium", "large"]}).fit(df)
out = enc.transform(df)
out[["size", "size_ord"]].drop_duplicates().sort_values("size_ord")

The order is an **argument**, not something inferred from the data. That single design
choice is what separates this from `factorize`: the order is stated once, stored in the
encoder, and serialized with it — so serve time cannot disagree with train time.

In [ ]:
# Round-trip through JSON, exactly as the bundle will do it.
restored = OrdinalEncoder.from_dict(json.loads(json.dumps(enc.to_dict())))

# Reversed rows, one level missing entirely — the codes do not budge.
odd_frame = pd.DataFrame({"size": ["large", "small"]})
print(restored.transform(odd_frame)[["size", "size_ord"]].to_dict("records"))

## 5. Strict at fit, lenient at predict

Two different situations that deserve two different answers:

- A value in the **fit** data that you forgot to declare is a mistake — a typo, a level
  you didn't know existed. Encoding it silently would invent a rank for it. → **raise**.
- A category first seen at **predict** time is normal life. → map it to `-1`, below every
  declared level, so a single tree split can isolate "unknown".

In [ ]:
try:
    OrdinalEncoder({"size": ["small", "medium"]}).fit(df)   # "large" not declared
except ValueError as e:
    print("fit raised:", e)

unseen = pd.DataFrame({"size": ["medium", "gigantic"]})
print(enc.transform(unseen)[["size", "size_ord"]].to_dict("records"))

## 6. Target encoding — the one that can leak

`city` is nominal, so ordinal is out. `FrequencyEncoder` is safe and cheap. `TargetEncoder`
is stronger *and* is the only encoder that reads `y`, so it gets special machinery.

The rule (see `docs/oof_vs_final_encoder.md`): the values you **train** on must be
computed **out-of-fold**, while the encoder you **serve** is fitted on all rows. Those
are two different objects, and `fit_encoding` hands you both.

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.encoders import fit_encoding as fit_encoding_lib

y = df["y"]
folds = make_folds(y, "binary", n_splits=3, seed=42)

oof_values, serving_encoder = fit_encoding_lib(df, ["city"], "target", y=y, folds=folds)

comparison = pd.DataFrame({
    "city": df["city"],
    "oof (what you train on)": oof_values["city_te"],
    "full fit (what you serve)": serving_encoder.transform(df)["city_te"],
})
comparison.drop_duplicates().head(8)

Same category, two different numbers. That gap **is** the honesty: the OOF value for a
row was computed without that row's fold, so a model fitted on it cannot memorise its own
target. The full-fit value is what a *new* row should get, because a new row was in no
fold at all.

**Check yourself:** if you trained on the full-fit column instead, would cross-validation
tell you something was wrong?

> No — it would tell you the model is *great*. Every row's feature would carry a smear of
> its own label, CV scores would go up, and the failure would only appear in production.
> That is exactly why this lives behind a tool instead of behind free-form code.

## 7. Now the tool layer

Everything above was the library. The **agent** never sees a DataFrame — tools take paths
and return a compact summary (§4). So `fit_encoding` as a tool:

- reads a table from disk,
- writes `tables/<id>.parquet` — the source table **plus** the new columns,
- writes `tables/<id>.pipeline.json` beside it: the **lineage**, i.e. the fitted encoders
  needed to rebuild those columns from raw input,
- records the table in the journal,
- returns paths and column names. No data.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import fit_encoding, inspect_data, train_tabular_model, finalize_run
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="y", n_splits=3),
)

profile = inspect_data(session)
print(profile["non_numeric_columns"])
print(profile["note"])

That note is the whole reason step 1 left the hole visible instead of papering over it:
the agent is *told* the columns exist and are unusable, and told which tool fixes it.

Now encode both columns — ordinal for `size`, target for `city` — chaining the second
call onto the table the first produced.

In [ ]:
step1 = fit_encoding(
    session, ["size"], method="ordinal",
    categories={"size": ["small", "medium", "large"]},
)
print(json.dumps(step1, indent=2))

In [ ]:
step2 = fit_encoding(session, ["city"], method="target", table=step1["table"])
print(step2["table"], "->", step2["new_columns"])
print(step2["note"])

`step2` was built **on top of** `t1`, so `t2` carries both encoders in its lineage. Two
tables now exist, and each knows exactly how it was made:

In [ ]:
from tabular_agent.harness import journal

for t in journal.load(session.run_dir)["tables"]:
    print(t["id"], "<-", t["source_table"], "|", t["method"], t["new_columns"], "| steps:", t["n_steps"])

In [ ]:
lineage = json.load(open(os.path.join(session.run_dir, "tables", "t2.pipeline.json")))
[(s["type"], s["encoder"]["kind"], s["encoder"]["columns"]) for s in lineage["steps"]]

### Why lineage per table, not one pipeline per run

A run branches: you might build `t1` and `t2`, then try a third table with different
encodings, and train models on several of them. A single run-wide pipeline would be a lie
about at least one of those models. Attaching the steps to the **table** means whichever
model wins, `finalize_run` can package exactly the transformations that produced *its*
features.

## 8. Does it actually help?

Train on the raw table (both categoricals dropped) and on `t2` (both encoded).

In [ ]:
raw_run = train_tabular_model(session, model_type="lgbm")
enc_run = train_tabular_model(session, table=step2["table"], model_type="lgbm")

print(f"raw     : {raw_run['cv_score']:.4f}  on {raw_run['n_features']} features, dropped {raw_run['dropped_non_numeric']}")
print(f"encoded : {enc_run['cv_score']:.4f}  on {enc_run['n_features']} features, dropped {enc_run['dropped_non_numeric']}")

The raw string columns are *still* reported as dropped — correct, and worth understanding:
`t2` contains `city`, `size`, `city_te` and `size_ord`. The originals stay for readability
and are ignored by the model; the encoded columns are what it trains on.

## 9. Finalize — the encoder has to survive into the bundle

This is where lineage pays off. `finalize_run` looks up which table the winning model was
trained on, loads that table's lineage, and makes it the bundle's pipeline.

In [ ]:
final = finalize_run(session)
print("best:", final["best_experiment_id"], "| cv:", round(final["cv_score"], 4))

In [ ]:
from tabular_agent.io import load_table
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(session.run_dir, final["bundle_dir"]))
print("pipeline steps:", [(s["type"], s["encoder"]["kind"]) for s in bundle["pipeline"].to_dict()["steps"]])
print("features      :", bundle["feature_names"])

In [ ]:
# Raw rows: strings, no encoded columns, no target. Exactly what production sends.
raw_rows = load_table(data_path).drop(columns=["y"]).head(5)
print(raw_rows.to_string(index=False))
print()
print("predictions:", predict_bundle(bundle, raw_rows))

Read that carefully, because it's the payoff for all the ceremony: the input had a
`size` string and a `city` string. The bundle re-derived `size_ord` from the **stored
level order** and `city_te` from the **full-fit** mapping, then predicted. Nothing about
the encoding depended on the rows we just passed in.

**Check yourself:** the `city_te` values used here differ from the ones in `t2.parquet`
that the model trained on. Is that a bug?

> No — it's the design. `t2.parquet` holds out-of-fold values (honest for training);
> the bundle holds the full-fit encoder (correct for new rows). Same column name, two
> deliberately different sources. `docs/oof_vs_final_encoder.md` has the long version.

## 10. The agent surface — now four tools

Every tool added is a hole punched in the "leakage-safe by construction" claim, so the
tool list is asserted in the tests, not just documented. `fit_encoding` is safe to expose
because the dangerous variant (`method="target"`) *cannot* be called without the fold
discipline — that's enforced in the function, not in the prompt.

In [ ]:
from tabular_agent.harness.agent import build_agent, DEFAULT_MODEL, DEFAULT_GATEWAY_MODEL

agent = build_agent()
print("tools:", sorted(agent._function_toolset.tools))

## 11. A live run — on a non-Anthropic model

Nothing in the harness is provider-specific. `gateway_model` points PydanticAI at any
OpenAI-compatible endpoint — here Requesty — so the whole loop runs on a model that has
nothing to do with the vendor whose SDK we started from.

The key is read from `config.env` (`REQUESTY_API_KEY=...`), which is gitignored. It is
never written into this notebook.

In [ ]:
from tabular_agent.harness.agent import Deps, gateway_model

model = gateway_model(DEFAULT_GATEWAY_MODEL)   # tensorx/deepseek-v4-flash
print("model:", DEFAULT_GATEWAY_MODEL, "| default (needs an Anthropic key):", DEFAULT_MODEL)

In [ ]:
fresh = RunSession(
    run_dir=os.path.join(workdir, "agent_run"),
    data_path=data_path,
    spec=TaskSpec(target="y", n_splits=3),
)

result = build_agent(model).run_sync(
    "Build the best model you can for this dataset, then finalize the run.",
    deps=Deps(session=fresh),
)
print(result.output)

In [ ]:
# What it actually did — the journal, not the model's own account of itself.
j = journal.load(fresh.run_dir)
print("tables     :", [(t["id"], t["method"], t["new_columns"]) for t in j.get("tables", [])])
print("leaderboard:", [(r["id"], r["model_type"], round(r["cv_score"], 4)) for r in j["leaderboard"]])
print("best       :", j["best"])

Read those two cells against each other. The model's prose is a *claim*; the journal is
the *record*, written as a side effect of the tools actually running. When they disagree,
the journal is right — which is the reason it exists (§19.2).

Two things worth checking in your own run:

- **Did it use `ordinal` for `size`?** When this notebook was written, it didn't — it
  frequency-encoded both columns. Nothing is broken by that (frequency encoding is safe),
  but it left signal on the table. The harness guarantees the agent *cannot do something
  unsafe*; it does not guarantee the agent does the *smartest* thing. Those are different
  properties, and only the first one can be enforced by construction.
- **How many experiments did it run?** It tends to keep tuning long after the gains stop.
  A harness-owned stop condition is on the list below.

## 12. Where this goes next

- `transform_features` — the sandboxed free-code escape hatch, the last tool that can
  create features. Its lineage step already exists (`Pipeline.add_transform`).
- `blend_models` and `predict`.
- A `journal_append` note tool so the agent can record *why*, not just *what*.